# Clase 4 — Datos y Sensores del Dominio Pesquero
## 🔴 Nivel AVANZADO — Datos reales, frentes térmicos e ingeniería de features

**Curso:** Inteligencia Artificial Aplicada a la Producción Pesquera
**Institución:** UTN FRCh · PesquerosEnIA · 2026
**Docentes:** Ariel Giamportone · Soraya Corvalán

---

**Para quién:** cómodo con Python/pandas y con nociones de ML. Este notebook va más allá
de la exploración: **ingesta de datos reales** (Copernicus / NOAA / xarray-NetCDF),
**detección de frentes térmicos** por gradiente espacial, **ingeniería de features**
ambientales para modelado, y una discusión de **calidad y sesgos** de los datos.

> Los datos sintéticos se usan como *fallback* cuando no hay credenciales/conexión, para
> que el notebook corra en cualquier entorno (incluido Colab sin configurar).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Configuración de estilo para gráficos
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')
np.random.seed(42)  # Reproducibilidad

print('✓ Librerías cargadas correctamente')
print(f'  NumPy {np.__version__} | Pandas {pd.__version__}')

## 1. Ingesta de datos reales (con *fallback* sintético)

En producción, la SST y la clorofila se obtienen de servicios operacionales. El patrón
robusto es **intentar la fuente real y degradar a datos sintéticos** si no está disponible,
para que el pipeline nunca se rompa.

In [ ]:
# ── Acceso a datos reales de SST (ejecutable si hay credenciales/conexión) ─────
def cargar_sst_copernicus():
    """Intenta abrir SST L4 de Copernicus Marine. Devuelve un xarray.Dataset o None."""
    try:
        import copernicusmarine as cm  # pip install copernicusmarine
        ds = cm.open_dataset(
            dataset_id="cmems_obs-sst_glo_sst_l4_nrt_observations_010_001",
            variables=["analysed_sst"],
            minimum_longitude=-65, maximum_longitude=-44,
            minimum_latitude=-55, maximum_latitude=-34,
        )
        print("✓ SST real de Copernicus cargada")
        return ds
    except Exception as e:
        print(f"[info] Copernicus no disponible en este entorno ({type(e).__name__}).")
        print("      → Continuamos con datos sintéticos representativos.")
        return None

ds_real = cargar_sst_copernicus()

# Alternativa sin credenciales: NOAA ERDDAP vía erddapy (griddap)
#   from erddapy import ERDDAP
#   e = ERDDAP(server="https://coastwatch.pfeg.noaa.gov/erddap", protocol="griddap")
#   e.dataset_id = "jplMURSST41"; ...  ->  e.to_xarray()


In [ ]:
# ── Patrón xarray/NetCDF: el estándar de datos oceanográficos ─────────────────
try:
    import xarray as xr
    print("xarray disponible:", xr.__version__)
    # Uso típico con un archivo .nc local:
    #   ds = xr.open_dataset('sst_pca.nc')
    #   sst_media = ds['analysed_sst'].mean(dim='time') - 273.15   # K -> °C
    #   sst_media.plot(cmap='RdYlBu_r')
except ImportError:
    print("xarray no instalado → pip install xarray netCDF4")


## 2. Grilla de SST y detección de frentes térmicos

Los **frentes** (gradientes fuertes de SST) concentran nutrientes y recursos. Se detectan
computando la **magnitud del gradiente espacial** de la SST — un feature clásico en
oceanografía pesquera.

In [ ]:
# ── Parámetros geográficos de la PCA ──────────────────────────────────────────
# Latitudes: de Tierra del Fuego (-55°S) a Buenos Aires (-34°S)
# Longitudes: plataforma continental (-65°O a -44°O)
latitudes = np.arange(-55, -34, 0.5)
longitudes = np.arange(-65, -44, 0.5)
lon_grid, lat_grid = np.meshgrid(longitudes, latitudes)

# ── SST media anual: gradiente latitudinal realista ───────────────────────────
# Norte (~-35°S): ~18°C | Sur (~-55°S): ~5°C
# Pendiente: ~0.65°C por grado de latitud
sst_base = 18 + 0.65 * lat_grid

# Variabilidad espacial: efecto de frente Malvinas-Brasil cerca del talud
ruido_espacial = np.random.normal(0, 0.8, lon_grid.shape)
gradiente_lon = 0.05 * (lon_grid + 52)  # frente cerca de la isobata de 200m
sst_media_anual = sst_base + ruido_espacial + gradiente_lon

print(f'Grid generado: {lat_grid.shape[0]} latitudes × {lon_grid.shape[1]} longitudes')
print(f'SST media: {sst_media_anual.mean():.1f}°C | Mín: {sst_media_anual.min():.1f}°C | Máx: {sst_media_anual.max():.1f}°C')

In [ ]:
# ── Frentes térmicos: magnitud del gradiente espacial de SST ──────────────────
grad_lat, grad_lon = np.gradient(sst_media_anual)
gradiente_sst = np.sqrt(grad_lat**2 + grad_lon**2)   # |∇SST|

fig, axes = plt.subplots(1, 2, figsize=(15, 7))
c0 = axes[0].contourf(longitudes, latitudes, sst_media_anual, levels=20, cmap='RdYlBu_r')
plt.colorbar(c0, ax=axes[0]).set_label('SST (°C)')
axes[0].set_title('SST media anual')
axes[0].set_xlabel('Longitud (°O)'); axes[0].set_ylabel('Latitud (°S)')

c1 = axes[1].contourf(longitudes, latitudes, gradiente_sst, levels=20, cmap='magma')
plt.colorbar(c1, ax=axes[1]).set_label('|∇SST| (°C / paso de grilla)')
axes[1].set_title('Frentes térmicos (gradiente de SST)\nvalores altos = frentes = agregación de recursos')
axes[1].set_xlabel('Longitud (°O)'); axes[1].set_ylabel('Latitud (°S)')
plt.tight_layout(); plt.show()

umbral_frente = np.percentile(gradiente_sst, 90)
print(f'Umbral de frente (P90 del gradiente): {umbral_frente:.3f} °C/paso')
print(f'Celdas clasificadas como frente: {(gradiente_sst >= umbral_frente).sum()} '
      f'({(gradiente_sst >= umbral_frente).mean():.1%} de la grilla)')


In [ ]:
# ── Clorofila-a: distribución espacial en la PCA ──────────────────────────────
np.random.seed(42)

# Alta productividad en la zona del frente (~-40°S) y golfos patagónicos
clorofila = np.abs(
    2.0 * np.exp(-((lat_grid + 40) ** 2) / 20)  # pico en el frente ~-40°S
    + 1.5 * np.exp(-((lat_grid + 43) ** 2) / 15)  # golfo San Jorge
    + 0.8 * np.exp(-((lat_grid + 47) ** 2) / 10)  # golfo San Matías
    + np.random.exponential(0.3, lon_grid.shape)   # ruido realista
)

# Figura comparativa: SST vs Clorofila
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# SST
c1 = axes[0].contourf(longitudes, latitudes, sst_media_anual, levels=15, cmap='RdYlBu_r')
plt.colorbar(c1, ax=axes[0]).set_label('SST (°C)')
axes[0].set_title('Temperatura Superficial del Mar (SST)', fontweight='bold')
axes[0].set_xlabel('Longitud (°O)')
axes[0].set_ylabel('Latitud (°S)')

# Clorofila
c2 = axes[1].contourf(longitudes, latitudes, clorofila, levels=15, cmap='YlGn')
plt.colorbar(c2, ax=axes[1]).set_label('Clorofila-a (mg/m³)')
axes[1].set_title('Clorofila-a (productividad primaria)', fontweight='bold')
axes[1].set_xlabel('Longitud (°O)')
axes[1].set_ylabel('Latitud (°S)')

plt.suptitle('PCA: SST vs Clorofila-a — indicadores de zonas de pesca',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 3. Ingeniería de features ambientales para modelado

Preparamos la tabla que alimentaría un modelo (Clase 6): a las variables crudas les
sumamos **features derivadas** — anomalía de SST, distancia al óptimo térmico y una
proxy de intensidad de frente por marea.

In [ ]:
# ── Dataset de capturas históricas (partes de pesca simulados) ─────────────────
np.random.seed(42)
n_mareas = 500

# Variables oceanográficas al momento de la marea
sst_marea = np.random.normal(10, 3.5, n_mareas)   # SST en zona de pesca
chl_marea = np.abs(np.random.exponential(1.8, n_mareas))  # Clorofila
profundidad_m = np.random.uniform(60, 280, n_mareas)
mes = np.random.randint(1, 13, n_mareas)
lat_captura = np.random.uniform(-50, -39, n_mareas)
lon_captura = np.random.uniform(-62, -50, n_mareas)

# Captura de merluza (tn) — función de SST con óptimo en 8-12°C
prob_exito_sst = np.exp(-((sst_marea - 10) ** 2) / 18)
prob_exito_chl = np.clip(chl_marea / 4, 0, 1)
prob_exito_prof = np.exp(-((profundidad_m - 140) ** 2) / 6000)

captura_base = 60 * (0.45 * prob_exito_sst + 0.3 * prob_exito_chl + 0.25 * prob_exito_prof)
captura_merluza_tn = np.abs(captura_base + np.random.normal(0, 10, n_mareas))

registros_captura = pd.DataFrame({
    'mes': mes,
    'lat_captura': lat_captura,
    'lon_captura': lon_captura,
    'profundidad_m': profundidad_m,
    'sst_grados': sst_marea,
    'clorofila_mg_m3': chl_marea,
    'captura_merluza_tn': captura_merluza_tn
})

# Agregar estación
registros_captura['estacion'] = pd.cut(
    registros_captura['mes'],
    bins=[0, 3, 6, 9, 12],
    labels=['Verano', 'Otoño', 'Invierno', 'Primavera'])

print(f'Partes de pesca generados: {len(registros_captura)}')
print(f'Captura promedio por marea: {registros_captura["captura_merluza_tn"].mean():.1f} tn')
registros_captura.describe().round(2)

In [ ]:
# ── Feature engineering sobre los registros de captura ────────────────────────
df = registros_captura.copy()

# 1) Anomalía de SST respecto de la media de la flota
df['sst_anomalia'] = df['sst_grados'] - df['sst_grados'].mean()
# 2) Distancia al óptimo térmico de la merluza (~10 °C) — no lineal
df['dist_optimo_termico'] = (df['sst_grados'] - 10).abs()
# 3) Índice de hábitat combinado (SST templada + clorofila alta)
df['indice_habitat'] = (
    np.exp(-((df['sst_grados'] - 10) ** 2) / 18) * np.clip(df['clorofila_mg_m3'] / 4, 0, 1)
)

nuevas = ['sst_anomalia', 'dist_optimo_termico', 'indice_habitat']
print('Features derivadas agregadas:', nuevas)
print(df[nuevas + ['captura_merluza_tn']].corr()['captura_merluza_tn'].round(3).to_string())
print('\n→ `indice_habitat` suele correlacionar mejor con la captura que las variables crudas: '
      'esa es la ganancia del feature engineering.')
df[['sst_grados', 'clorofila_mg_m3'] + nuevas + ['captura_merluza_tn']].head()


In [ ]:
# ── Correlación entre variables ambientales y captura ─────────────────────────
variables_num = ['sst_grados', 'clorofila_mg_m3', 'profundidad_m', 'captura_merluza_tn']
correlaciones = registros_captura[variables_num].corr()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap de correlación
sns.heatmap(correlaciones, annot=True, fmt='.2f', cmap='RdYlBu',
            center=0, ax=axes[0], square=True,
            xticklabels=['SST (°C)', 'Clorofila', 'Profundidad', 'Captura (tn)'],
            yticklabels=['SST (°C)', 'Clorofila', 'Profundidad', 'Captura (tn)'])
axes[0].set_title('Correlaciones entre variables\nambientales y captura', fontsize=12)

# SST óptima para captura (binned)
bins_sst = pd.cut(registros_captura['sst_grados'], bins=8)
captura_por_sst = registros_captura.groupby(bins_sst, observed=True)['captura_merluza_tn'].mean()
captura_por_sst.plot(kind='bar', ax=axes[1], color='steelblue', alpha=0.8, edgecolor='white')
axes[1].set_xlabel('Rango de SST (°C)')
axes[1].set_ylabel('Captura promedio (tn)')
axes[1].set_title('Captura de merluza por rango de SST\n(óptimo ~8-12°C)', fontsize=12)
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print('\nCorrelación SST - Captura:', correlaciones.loc['sst_grados', 'captura_merluza_tn'].round(3))
print('Correlación Clorofila - Captura:', correlaciones.loc['clorofila_mg_m3', 'captura_merluza_tn'].round(3))

## 4. Calidad de datos y sesgos (lo que no se ve en el gráfico)

Antes de modelar, hay que auditar los datos:

- **Dark vessels:** barcos que apagan el AIS. El esfuerzo pesquero de GFW **subestima**
  la actividad real; se corrige fusionando **AIS + SAR** (radar satelital).
- **Gaps temporales:** las imágenes ópticas de SST/clorofila fallan con **nubosidad**.
  Soluciones: productos *L4* interpolados o *gap-filling* (DINEOF, kriging).
- **Resolución:** ~5–9 km puede no capturar frentes finos costeros; evaluar productos de 1 km.
- **Sesgo de muestreo en capturas:** los partes reflejan **dónde se pescó**, no dónde
  **hay** recurso (sesgo de esfuerzo). Cuidado al entrenar modelos con este sesgo.
- **Referenciación temporal:** siempre unir la captura con la variable ambiental **del mismo día/zona**,
  no con la media climatológica.

## 🔴 Desafío avanzado

1. **Datos reales:** registrate en Copernicus Marine y descargá 1 semana de `analysed_sst`
   para la PCA. Calculá la **anomalía** respecto de la climatología mensual.
2. **Frentes en el tiempo:** computá `|∇SST|` para cada día y detectá cómo **migra el frente**
   Malvinas-Brasil a lo largo de la semana.
3. **Cruce con esfuerzo:** bajá el *fishing effort* diario de Global Fishing Watch para la
   misma zona/fecha y evaluá la correlación **frente ↔ esfuerzo pesquero**.
4. **Discusión:** ¿el esfuerzo sigue al frente, o hay retardo? ¿Qué implica para predecir
   zonas (Clase 6)?

*Entregable sugerido: un notebook con el pipeline reproducible y una figura frente-vs-esfuerzo.*

## ✅ Síntesis (nivel avanzado)

- El patrón **fuente-real-con-fallback** hace pipelines robustos y portables.
- `xarray`/NetCDF es el estándar para datos oceanográficos multidimensionales.
- El **gradiente de SST** es un detector de frentes barato y potente.
- El **feature engineering** ambiental (anomalías, índice de hábitat) suele superar a las
  variables crudas para modelar captura.
- La **calidad y el sesgo** de los datos condicionan todo modelo posterior.

### Referencias técnicas
- Copernicus Marine Toolbox: https://marine.copernicus.eu
- erddapy (NOAA ERDDAP): https://ioos.github.io/erddapy
- xarray: https://tutorial.xarray.dev
- Global Fishing Watch API: https://globalfishingwatch.org/our-apis
- Kroodsma et al. 2018, *Science* 359(6378): 904-908 (AIS + fishing footprint)